# Module 4 — Code Along: JSON & Pydantic (the bank-accounts story)

Module 3 gave us an account as a `@dataclass` — but a dataclass trusts you (`balance=-50` sails through). Today we upgrade to a **Pydantic `BaseModel`**: same fields, but **validated on construction**, with clean dict↔model conversion — then wire it into FastAPI for automatic 422s.

Section by section, one code cell per beat — run top to bottom:

**Section 1** dataclass → Pydantic · **Section 2** validation rules · **Section 3** model_validate / model_dump · **Section 4** Pydantic in FastAPI

## Section 1 — From @dataclass to Pydantic

A `@dataclass` stores whatever you pass; a Pydantic `BaseModel` has the same fields but checks them when you build it.

In [ ]:
# 1.1 · A @dataclass trusts you — it stores nonsense silently.
from dataclasses import dataclass

@dataclass
class BankAccountDC:
    id: int
    owner: str
    balance: float

bad = BankAccountDC(id=1, owner="", balance=-50)   # accepted: empty owner, negative balance
print(bad)   # no error at all

In [ ]:
# 1.2 · Pydantic — subclass BaseModel, list the SAME typed fields.
from pydantic import BaseModel

class BankAccount(BaseModel):
    id: int
    owner: str
    balance: float

print(BankAccount(id=1, owner="Ada", balance=1500.0))   # a validated object

In [ ]:
# 1.3 · Validated on construction — good data builds; bad data raises immediately.
from pydantic import ValidationError

print(BankAccount(id=1, owner="Ada", balance=1500.0))    # ok
try:
    BankAccount(id="x", owner="Ada", balance=1500.0)     # id is not an int
except ValidationError as e:
    print("ValidationError:", e.errors()[0]["type"])     # int_parsing

## Section 2 — Validation rules

`Field(...)` adds rules beyond the type (e.g. `ge=0`, `min_length=1`). When they fail, the `ValidationError` is structured — it tells you which field and why.

In [ ]:
# 2.1 · Field — rules beyond the type, checked on construction.
from pydantic import Field

class BankAccount(BaseModel):
    id: int
    owner: str = Field(min_length=1)
    balance: float = Field(ge=0)        # ge = greater-than-or-equal

print(BankAccount(id=1, owner="Ada", balance=1500.0))   # passes the rules

In [ ]:
# 2.2 · A ValidationError is structured: where (loc), what (msg), why (type).
try:
    BankAccount(id=1, owner="", balance=-5)
except ValidationError as e:
    for err in e.errors():
        print(err["loc"], "->", err["type"])
# ('owner',) -> string_too_short
# ('balance',) -> greater_than_equal

## Section 3 — model_validate / model_dump

The two methods you use on every line of Modules 5–6: a plain dict in (validated), a plain dict out — and lax coercion means a CSV of strings just works.

In [ ]:
# 3.1 · model_validate — a plain dict (an HTTP body / file row) becomes a validated model.
acct = BankAccount.model_validate({"id": 2, "owner": "Lin", "balance": 800.0})
print(acct, "|", type(acct).__name__)

In [ ]:
# 3.2 · model_dump — the model back to a plain dict (ready for JSON / the wire).
print(acct.model_dump())   # {'id': 2, 'owner': 'Lin', 'balance': 800.0}

In [ ]:
# 3.3 · Coercion — Pydantic is lax: numeric/bool strings convert. This is why a CSV of
#        strings can feed the API with NO manual parsing (the Module 6 through-line).
c = BankAccount.model_validate({"id": "2", "owner": "Lin", "balance": "800.0"})
print(type(c.id).__name__, type(c.balance).__name__, "->", c.id, c.balance)   # int float -> 2 800.0

## Section 4 — Pydantic in FastAPI

Type a route's body as your model and FastAPI validates it for you (automatic 422); `response_model` types the output; the model becomes the `/docs` schema. We drive it in-process with TestClient.

In [ ]:
# 4.1 · A typed body validates itself — bad JSON returns 422 before your code runs.
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()
ACCOUNTS = {}

@app.post("/accounts", status_code=201)
def create(account: BankAccount):      # FastAPI validates the body INTO a BankAccount
    ACCOUNTS[account.id] = account.model_dump()
    return account

client = TestClient(app)
print(client.post("/accounts", json={"id": 1, "owner": "Ada", "balance": 1500.0}).status_code)  # 201
print(client.post("/accounts", json={"id": 2, "owner": "", "balance": -5}).status_code)          # 422

In [ ]:
# 4.2 · response_model — typed output; FastAPI serialises the model to JSON.
@app.get("/accounts/{id}", response_model=BankAccount)
def get_account(id: int):
    return ACCOUNTS[id]

print(client.get("/accounts/1").json())   # {'id': 1, 'owner': 'Ada', 'balance': 1500.0}

In [ ]:
# 4.3 · The model IS the schema — /docs is generated from it. The OpenAPI schema proves it:
schema = client.get("/openapi.json").json()
print("BankAccount" in schema["components"]["schemas"])   # True — fields/types/constraints documented

## Next: Module 5

`BankAccount` is now validated, and `model_validate` / `model_dump` move it between dicts and objects.

**Next we drive the Day-1 server from Python** — an `APIClient` over `requests` that calls the API and parses each response back into these Pydantic models.